# Verify game controller

Use this notebook to verify the controller connection and logical input mapping without connecting to or moving the robot. Press each required button, D-pad direction, stick, and trigger; exit with **select/back** or interrupt the cell.

In [ ]:
import sys
import time
from importlib.metadata import PackageNotFoundError, version

print(f'Python: {sys.version.split()[0]}')
try:
    print(f'approxeng.input: {version("approxeng.input")}')
except PackageNotFoundError:
    print('FAIL  approxeng.input is not installed in this kernel.')

if sys.version_info[:2] != (3, 10):
    print('NOTE  The documented Voyee 360 setup uses approxeng.input 2.5 with Python 3.10.')

from approxeng.input.selectbinder import ControllerResource, ControllerNotFoundError

In [ ]:
# Observe controls for as long as needed. This talks only to the game controller.
required_buttons = {'square', 'triangle', 'circle', 'cross', 'l1', 'r1', 'select'}
required_dpad = {'dleft', 'dright', 'dup', 'ddown'}
seen = set()

try:
    with ControllerResource() as joystick:
        print('PASS  Controller connected.')
        print('Available controls:', joystick.controls)
        print('Move both sticks and triggers; press every button and D-pad direction. Select/back exits.')
        while joystick.connected:
            presses = joystick.check_presses()
            if presses.names:
                seen.update(presses.names)
                print('Pressed:', presses.names)
            axes = {name: getattr(joystick, name, None) for name in ('lx', 'ly', 'rx', 'ry', 'lt', 'rt')}
            active_axes = {name: round(value, 3) for name, value in axes.items()
                           if value is not None and abs(value) > 0.05}
            if active_axes:
                print('Axes:', active_axes)
            for direction in required_dpad:
                if joystick[direction] is not None:
                    seen.add(direction)
            if 'select' in presses.names or 'back' in presses.names:
                break
            time.sleep(0.02)
except ControllerNotFoundError as error:
    print('FAIL  Controller not found.')
    print(error)
except KeyboardInterrupt:
    print('Controller monitor interrupted by user.')

missing = (required_buttons | required_dpad) - seen
print('Observed controls:', sorted(seen))
print('PASS  All requested digital controls were observed.' if not missing else f'INCOMPLETE  Not observed: {sorted(missing)}')

A successful test shows the expected controller identity and control names, produces non-zero values for all sticks/triggers, and records every required digital control. If the controller is not found, check its USB/Bluetooth connection and the kernel's `approxeng.input` installation before trying again.